### Mass-Spring Lattice (2D extension of the chain model)

The system extends to a 2D square lattice of point masses, each of mass $m$, connected by ideal springs with spring constant $k$ and zero rest length. The boundaries are fixed. For a grid of size $N \times N$, with equilibrium positions at $(x_{ij}^0, y_{ij}^0) = (i a, j a)$ for $i,j = 0,\dots,N-1$, we define displacements $(u_{ij}, v_{ij}) = (x_{ij} - x_{ij}^0, y_{ij} - y_{ij}^0)$ for interior points $i,j = 1,\dots,N-2$. The equations of motion decouple for $u$ and $v$:

\begin{cases}
    m \frac{d^2 u_{ij}}{dt^2} + k\left(4u_{ij} - u_{i-1,j} - u_{i+1,j} - u_{i,j-1} - u_{i,j+1}\right) = 0 \\
    m \frac{d^2 v_{ij}}{dt^2} + k\left(4v_{ij} - v_{i-1,j} - v_{i+1,j} - v_{i,j-1} - v_{i,j+1}\right) = 0
\end{cases}

In matrix form, defining vectors $\mathbf{u}$ and $\mathbf{v}$ by stacking all $u_{ij}$ and $v_{ij}$ in row-major order, the system becomes:

$$
    \frac{d^2}{dt^2}\begin{bmatrix} \mathbf{u} \\ \mathbf{v} \end{bmatrix} + \frac{k}{m} \begin{bmatrix} \mathbf{K}_{2D} & \mathbf{0} \\ \mathbf{0} & \mathbf{K}_{2D} \end{bmatrix} \begin{bmatrix} \mathbf{u} \\ \mathbf{v} \end{bmatrix} = \mathbf{0},
$$

where $\mathbf{K}_{2D}$ is the discrete 2D Laplacian matrix with Dirichlet boundary conditions.

In [1]:
# Importing libraries
import numpy as np
from scipy.sparse.linalg import LaplacianNd
from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

In [2]:
def K_matrix_2D(N):

    return LaplacianNd(
        grid_shape = (N - 2, N - 2),  # Only interior points
        boundary_conditions = 'dirichlet',
        dtype = np.float64
    )

def initial_2D(N, a):

    num_interior = (N - 2) ** 2

    # Random displacements for both x and y directions
    u_ini = (np.random.rand(num_interior) - 0.5) * 0.4 * a
    v_ini = (np.random.rand(num_interior) - 0.5) * 0.4 * a
    return (u_ini, v_ini)

def system_rhs_2D(t, y, k, m, K):

    num_interior = len(y) // 4  # u, v, du/dt, dv/dt

    u = y[0 : num_interior]
    v = y[num_interior : 2 * num_interior]

    du_dt = y[2 * num_interior : 3 * num_interior]
    dv_dt = y[3 * num_interior : 4 * num_interior]
    
    # Accelerations from linearized equations
    acc_u = (k / m) * (K @ u)
    acc_v = (k / m) * (K @ v)
    
    dydt = np.concatenate([du_dt, dv_dt, acc_u, acc_v])
    return dydt

def solver_2D(N, m, k, a, t_span):

    K = K_matrix_2D(N)
    u_ini, v_ini = initial_2D(N, a)
    
    # Initial state: [u, v, du/dt, dv/dt]
    num_interior = (N - 2) ** 2
    
    y0 = np.concatenate([
        u_ini, v_ini, 
        np.zeros(num_interior), 
        np.zeros(num_interior)
    ])
    
    solution = solve_ivp(
        fun = lambda t, y: system_rhs_2D(t, y, k, m, K),
        t_span = t_span,
        y0 = y0,
        dense_output = True
    )
    
    return solution, K

def create_animation_2D(solution, N, a, output_file = 'Spring_Mass_System_2D.gif'):

    fps, dpi = 60, 120
    
    # Extract data
    t = solution.t
    y = solution.y
    num_interior = (N - 2) ** 2
    
    # Separate displacements
    u_history = y[0 : num_interior, :]
    v_history = y[num_interior : 2 * num_interior, :]
    
    # Equilibrium positions
    X_eq, Y_eq = np.meshgrid(
        np.arange(1, N - 1) * a,
        np.arange(1, N - 1) * a
    )
    X_eq = X_eq.flatten()
    Y_eq = Y_eq.flatten()
    
    fig, ax = plt.subplots(figsize = (8, 8))
    ax.set_xlim(- 0.5, (N - 1) * a + 0.5)
    ax.set_ylim(- 0.5, (N - 1) * a + 0.5)
    ax.set_aspect('equal')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('2D Mass-Spring Lattice Oscillations')
    
    # Plot fixed boundary points
    boundary_x, boundary_y = [], []
    for i in [0, N - 1]:
        for j in range(N):
            boundary_x.append(i * a)
            boundary_y.append(j * a)
    for j in [0, N - 1]:
        for i in range(1, N - 1):
            boundary_x.append(i * a)
            boundary_y.append(j * a)
    
    ax.plot(boundary_x, boundary_y, 'ks', markersize = 6, alpha = 0.7)
    
    # Plot movable points
    points = ax.plot([], [], 'ro', markersize = 5, alpha = 0.8)[0]
    
    def init():
        points.set_data(X_eq, Y_eq)
        return (points,)
    
    def update(frame):
        X_current = X_eq + u_history[:, frame]
        Y_current = Y_eq + v_history[:, frame]
        points.set_data(X_current, Y_current)
        return (points,)
    
    ani = FuncAnimation(
        fig = fig,
        func = update,
        frames = len(t),
        init_func = init,
        blit = True,
        interval = 1000 / fps,
        cache_frame_data = False
    )
    
    print(f"Creating GIF animation...")
    writer = PillowWriter(fps = fps, bitrate = 2_000)
    ani.save(
        output_file,
        writer = writer,
        dpi = dpi,
        progress_callback = lambda i, n: print(f"\rFrame {i + 1}/{n} processed...", end = '')
    )
    
    plt.close(fig)

    print("Done!")
    return None

In [3]:
# Solve and animate the 2D system
N = 15 
m = 1.5
k = 3.0
a = 0.25
t_span = (0, 20)

solution_2D, K_2D = solver_2D(N, m, k, a, t_span)

print(f"System solved successfully!")
print(f"Grid size: {N}x{N}")
print(f"Number of interior points: {(N - 2) ** 2}")
print(f"Number of time steps: {len(solution_2D.t)}")

create_animation_2D(solution_2D, N, a)

System solved successfully!
Grid size: 15x15
Number of interior points: 169
Number of time steps: 79
Creating GIF animation...
Frame 79/79 processed...Done!


In [4]:
eigs_2D = (- K_2D.eigenvalues())[::-1]

print("Eigenvalues of K_2D (first 10):", eigs_2D[0 : 10])
print("\nNormal frequencies (first 10):", np.sqrt(eigs_2D[0 : 10] * k / m))

Eigenvalues of K_2D (first 10): [0.10028835 0.24820644 0.24820644 0.39612453 0.48648121 0.48648121
 0.6343993  0.6343993  0.80316457 0.80316457]

Normal frequencies (first 10): [0.4478579  0.70456574 0.70456574 0.89008374 0.98638858 0.98638858
 1.12640961 1.12640961 1.26741041 1.26741041]
